# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [14]:
%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [15]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [21]:
import os
from glob import glob

price_data = os.environ.get('PRICE_DATA', '../../05_src/data/prices/')
parquet_files = glob(os.path.join(price_data, '**', '*.parquet'), recursive=True)

print(parquet_files)

['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2009\\part.0.parque

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [26]:
ddf = dd.read_parquet(parquet_files, engine='pyarrow', index=False)
print(ddf.columns)

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year'],
      dtype='object')


In [31]:
ddf = ddf.sort_values(['ticker', 'Date'])

# Define feature engineering function
def add_features(df):
    df = df.sort_values(['ticker', 'Date'])  # Re-sort to be safe
    df['Close_lag_1'] = df.groupby('ticker')['Close'].shift(1)
    df['Adj_Close_lag_1'] = df.groupby('ticker')['Adj Close'].shift(1)
    df['returns'] = (df['Close'] / df['Close_lag_1']) - 1
    df['hi_lo_range'] = df['High'] - df['Low']
    return df

# Apply the transformation partition-wise
dd_feat = ddf.map_partitions(add_features)


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [33]:
df_feat = dd_feat.compute()

df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].rolling(10).mean().reset_index(level=0, drop=True)


In [35]:
# Show the first few rows
print(df_feat[['ticker', 'returns', 'returns_ma_10']].head(15))


   ticker   returns  returns_ma_10
0     ACN       NaN            NaN
1     ACN -0.010547            NaN
2     ACN -0.000666            NaN
3     ACN -0.009333            NaN
4     ACN  0.006057            NaN
5     ACN -0.030100            NaN
6     ACN  0.000690            NaN
7     ACN  0.013094            NaN
8     ACN  0.017687            NaN
9     ACN  0.036096            NaN
10    ACN -0.006452       0.001653
11    ACN -0.016234       0.001084
12    ACN -0.003300       0.000820
13    ACN  0.026490       0.004403
14    ACN -0.032258       0.000571


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [ ]:
# It was not strictly necessary to convert to pandas to calculate the moving average return since dask also supports rolling operations. Not in this case, for small to medium datasets, 
# it is easier to use pandas. Also, for Daskmore data treatment would be required to make sure that the data is partitioned correctly for rolling operations.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.